# EEG Cognitive Stress Classification

## Data Loading and Exploration

This script loads EEG recording, checks for problems, and creates visualizations that build intuition about what EEG signals look like and how they differ between stressed vs relaxed states.


DATASET: SAM 40
- 40 subjects, 32 EEG channels, 128 Hz sampling rate
- 4 tasks (Relaxation, Stroop, Arithmetic, Mirror Image)
- 3 trials per task per subject = 480 total recordings
- Each recording = 25 seconds of brain activity

In [1]:
# Imports:

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # Using non-interactive backend (works everywhere)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.io import loadmat
from glob import glob
import warnings
warnings.filterwarnings('ignore')
 
# Setting a clean plotting style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

In [20]:
# Path location:

DATA_DIR = os.path.join("..", "Data", "SAM40")
print(DATA_DIR)

OUTPUT_DIR = os.path.join("..", "Results", "phase1")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(OUTPUT_DIR)

..\Data\SAM40
..\Results\phase1


Constants:

The 32 EEG channel names, in the exact order stored in the .mat files.
These follow the 10-20 international system for electrode placement.

Naming convention:
- Letter = brain region:
    - Fp = Frontopolar (forehead)
    - F  = Frontal
    - FC = Fronto-Central
    - FT = Fronto-Temporal
    - C  = Central
    - T  = Temporal
    - CP = Centro-Parietal
    - P  = Parietal
    - PO = Parieto-Occipital 
    - O  = Occipital
- Number = hemisphere:
    - z  = midline (center)
    - odd numbers = left hemisphere
    - even numbers = right hemisphere

In [3]:


CHANNEL_NAMES = [
    'Cz', 'Fz', 'Fp1', 'F7', 'F3', 'FC1', 'C3', 'FC5',
    'FT9', 'T7', 'CP5', 'CP1', 'P3', 'P7', 'PO9', 'O1',
    'Pz', 'Oz', 'O2', 'PO10', 'P8', 'P4', 'CP2', 'CP6',
    'T8', 'FT10', 'FC6', 'C4', 'FC2', 'F4', 'F8', 'Fp2'
]
 
SAMPLING_RATE = 128       # Samples per second (Hz)
TRIAL_DURATION = 25       # Each recording is 25 seconds long
EXPECTED_SAMPLES = 3200   # 128 Hz × 25 s = 3200 samples
N_CHANNELS = 32
N_SUBJECTS = 40
N_TRIALS = 3
 
# The 4 experimental tasks
TASK_NAMES = ['Relaxation', 'Stroop', 'Mirror_Image', 'Arithmetic']
 
# Colors for each task (used consistently across all plots)
TASK_COLORS = {
    'Relaxation':   '#2ecc71',   # Green — calm state
    'Stroop':       '#e74c3c',   # Red — high cognitive interference
    'Mirror_Image': '#f39c12',   # Orange — spatial reasoning
    'Arithmetic':   '#3498db',   # Blue — mental computation
}
 
# Self-reported stress ratings from Table 1 of the SAM 40 paper (Data in Brief).
# Format: subject_id -> [[T1_stroop, T1_arith, T1_mirror],
#                         [T2_stroop, T2_arith, T2_mirror],
#                         [T3_stroop, T3_arith, T3_mirror]]
STRESS_RATINGS = {
    1:[[3,6,3],[2,7,5],[4,4,7]], 2:[[5,3,4],[4,3,4],[3,7,5]],
    3:[[4,5,3],[5,3,5],[5,8,7]], 4:[[4,5,3],[2,3,5],[5,7,5]],
    5:[[6,6,6],[2,5,3],[3,5,7]], 6:[[2,5,4],[3,4,3],[5,8,6]],
    7:[[4,5,5],[3,3,3],[6,6,3]], 8:[[4,3,5],[3,6,6],[3,5,6]],
    9:[[4,3,6],[4,4,4],[6,7,4]], 10:[[4,3,4],[3,4,5],[5,6,4]],
    11:[[4,7,3],[5,6,5],[5,5,6]], 12:[[2,5,7],[5,5,4],[3,8,6]],
    13:[[3,3,5],[2,3,4],[1,3,5]], 14:[[3,3,4],[2,6,4],[4,6,3]],
    15:[[5,9,6],[3,7,5],[5,6,5]], 16:[[1,4,4],[1,5,6],[6,8,5]],
    17:[[8,8,9],[4,6,8],[6,7,7]], 18:[[6,6,5],[5,5,6],[7,9,6]],
    19:[[6,4,6],[3,2,4],[3,4,1]], 20:[[7,10,9],[5,10,9],[6,10,8]],
    21:[[9,7,8],[9,8,8],[9,7,8]], 22:[[5,9,8],[6,8,8],[5,7,6]],
    23:[[1,3,2],[1,4,2],[5,8,4]], 24:[[1,4,2],[1,2,1],[4,5,7]],
    25:[[2,2,1],[2,2,1],[5,5,7]], 26:[[8,6,7],[7,6,8],[6,4,5]],
    27:[[3,5,5],[2,3,4],[3,5,3]], 28:[[7,7,8],[5,7,7],[4,6,5]],
    29:[[6,6,7],[6,8,7],[5,6,6]], 30:[[5,7,6],[5,8,6],[4,7,5]],
    31:[[4,10,8],[3,9,7],[7,6,3]], 32:[[1,5,5],[1,1,2],[3,7,6]],
    33:[[3,5,4],[2,3,2],[6,8,3]], 34:[[1,2,3],[1,3,1],[3,2,2]],
    35:[[1,6,5],[1,5,2],[5,4,6]], 36:[[6,4,4],[3,5,4],[5,5,5]],
    37:[[4,6,5],[2,5,3],[6,8,4]], 38:[[5,6,4],[3,5,6],[3,4,5]],
    39:[[3,6,4],[3,5,5],[5,6,3]], 40:[[5,3,5],[5,4,6],[3,4,4]],
}

## Understanding .MAT File Format

How .MAT File work?

A .mat file is MATLAB's way of saving variables. When we load it in Python using scipy.io.loadmat(), we get a Python dictionary where:
- Keys starting with '__' are MATLAB metadata (ignore these)
- Other keys are the actual data variables
    
For the SAM 40 dataset, the EEG data is stored as a 2D array:
- Rows = channels (32)
- Columns = time samples (~3200)
    
The data is in EEGLAB .set format, which is just a .mat file with specific variable names.


In [4]:

"""
Loading one .mat file and return the EEG data as a numpy array

Parameters:
    filepath: path to a .mat file
    
Returns:
    eeg_data: numpy array of shape (32, ~3200)
    data_key: the key name used to store the EEG data
"""

def load_single_mat(filepath):
    mat_contents = loadmat(filepath)
    
    # Filter out MATLAB metadata keys (starting with '__')
    data_keys = [k for k in mat_contents.keys() if not k.startswith('__')]
    
    # Find the largest array (EEG data)
    eeg_data = None
    data_key = None
    for key in data_keys:
        arr = mat_contents[key]
        if isinstance(arr, np.ndarray):
            # Handle nested MATLAB structures (EEGLAB format)
            if arr.dtype == np.object_:             # If MATLAB structures are loaded as object arrays in Numpy
                try:
                    inner = arr.flat[0]
                    if hasattr(inner, 'dtype') and inner.dtype.names:     # To check if it's a structured array (MATLAB struct) and check for a field named 'data'
                        if 'data' in inner.dtype.names:
                            arr = inner['data'].flat[0]
                except (IndexError, AttributeError):
                    continue
            
            if eeg_data is None or arr.size > (eeg_data.size if eeg_data is not None else 0):
                eeg_data = arr.astype(np.float64)         # Converts largest array to 64 bit floating point for numerical processing
                data_key = key      # Stores the name of the variable that holds the largest array (EEG data)
    
    return eeg_data, data_key




In [5]:


"""

    Extracting subject number, task name, and trial number from filename.
    
    Naming convention in the dataset:
        Relax_sub_1_trial1.mat             → Relaxation, Subject 1, Trial 1
        Stroop_sub_5_trial2.mat            → Stroop, Subject 5, Trial 2
        Mirror_image_sub_10_trial3.mat     → Mirror Image, Subject 10, Trial 3
        Arithmetic_sub_40_trial1.mat       → Arithmetic, Subject 40, Trial 1
    
    Pattern: Task_sub_N_trialM.mat
    
    Returns: subject_id (int), task_name (str), trial_num (int)
"""

def parse_filename(filename):
    name = filename.replace('.mat', '')
    parts = name.split('_')
    
    # Extract subject number (after 'sub')
    subject_id = int(parts[parts.index('sub') + 1])
    
    # Extract trial number - attached after trial like "trial1", "trial2"
    # Find the part that starts with "trial" and extract the number from it
    trial_part = [p for p in parts if p.startswith('trial')][0]
    trial_num = int(trial_part.replace('trial', ''))
    
    # Identify task from keywords in the filename
    fname_lower = filename.lower()
    if 'relax' in fname_lower:
        task_name = 'Relaxation'
    elif 'stroop' in fname_lower:
        task_name = 'Stroop'
    elif 'mirror' in fname_lower:
        task_name = 'Mirror_Image'
    elif 'arithmetic' in fname_lower:
        task_name = 'Arithmetic'
    else:
        task_name = 'Unknown'
    
    return subject_id, task_name, trial_num

## Loading Data

Loading every .mat file from the filtered_data folder into a structured dictionary.

The dataset provides both raw and filtered versions. The filtered data has already been cleaned by the researchers:
1. Band-pass filtered (0.5-45 Hz) to remove electrical noise
2. Baseline drift removed using Savitzky-Golay filter
3. Artifacts (eye blinks, muscle movements) removed using wavelet thresholding
    
Using the pre-filtered data lets us focus on feature extraction and classification without worrying about complex preprocessing.

In [6]:

def load_all_data(data_dir):
    # Returns:
    #     data_dict: {subject_id: {task_name: {trial_num: np.array(32, ~3200)}}}
    #     file_info: list of dicts with metadata about each loaded file

    filtered_dir = os.path.join(data_dir, 'filtered_data')
    if not os.path.exists(filtered_dir):
        print(f"ERROR: {filtered_dir} not found!")
        print(f"Make sure you extracted the dataset correctly.")
        return None, None
    
    mat_files = sorted(glob(os.path.join(filtered_dir, '*.mat')))
    print(f"Found {len(mat_files)} .mat files in filtered_data/")
    
    data_dict = {}
    file_info = []
    errors = []
    
    for i, filepath in enumerate(mat_files):
        filename = os.path.basename(filepath)
        
        # Show progress every 50 files
        if (i + 1) % 50 == 0 or i == 0:
            print(f"  Loading file {i+1}/{len(mat_files)}: {filename}")
        
        try:
            subject_id, task_name, trial_num = parse_filename(filename)
            eeg_data, data_key = load_single_mat(filepath)
            
            if eeg_data is None:
                errors.append(f"No data found in {filename}")
                continue
            
            # Store in nested dictionary
            if subject_id not in data_dict:
                data_dict[subject_id] = {}
            if task_name not in data_dict[subject_id]:
                data_dict[subject_id][task_name] = {}
            data_dict[subject_id][task_name][trial_num] = eeg_data
            
            # Record file metadata
            file_info.append({
                'filename': filename,
                'subject': subject_id,
                'task': task_name,
                'trial': trial_num,
                'n_channels': eeg_data.shape[0],
                'n_samples': eeg_data.shape[1] if eeg_data.ndim > 1 else len(eeg_data),
                'data_key': data_key,
                'min': eeg_data.min(),
                'max': eeg_data.max(),
                'mean': eeg_data.mean(),
                'std': eeg_data.std(),
            })
            
        except Exception as e:
            errors.append(f"{filename}: {str(e)}")
    
    file_info_df = pd.DataFrame(file_info)
    
    # Print summary
    print(f"\n{'='*60}")
    print(f"LOADING COMPLETE")
    print(f"{'='*60}")
    print(f"Subjects loaded:  {len(data_dict)}")
    print(f"Files loaded:     {len(file_info)}")
    print(f"Files expected:   {N_SUBJECTS * len(TASK_NAMES) * N_TRIALS}")
    
    if errors:
        print(f"\nErrors ({len(errors)}):")
        for e in errors[:5]:
            print(f"  {e}")
    
    return data_dict, file_info_df

## Exploring Structure

Examining the loaded data for consistency and potential issues.

Checking if:
- All files have 32 channels?
- All files have ~3200 samples (25 seconds)?
- there are any missing subjects/tasks/trials?
- What's the amplitude range of the signals?

In [7]:


def explore_structure(data_dict, file_info_df):

    print(f"\n{'='*60}")
    print(f"DATA STRUCTURE EXPLORATION")
    print(f"{'='*60}")
    
    # --- 1st Check: Channel counts ---
    unique_channels = file_info_df['n_channels'].unique()
    print(f"\n1. Channel counts: {unique_channels}")
    if len(unique_channels) == 1 and unique_channels[0] == N_CHANNELS:
        print(f"   ✓ All files have exactly {N_CHANNELS} channels")
    else:
        print(f"   ✗ WARNING: Inconsistent channel counts!")
    
    # --- 2nd Check: Sample counts ---
    print(f"\n2. Sample counts:")
    print(f"   Min:  {file_info_df['n_samples'].min()}")
    print(f"   Max:  {file_info_df['n_samples'].max()}")
    print(f"   Mean: {file_info_df['n_samples'].mean():.1f}")
    print(f"   Expected: {EXPECTED_SAMPLES}")
    
    n_exact = (file_info_df['n_samples'] == EXPECTED_SAMPLES).sum()
    print(f"   Files with exactly {EXPECTED_SAMPLES} samples: {n_exact}/{len(file_info_df)}")
    
    # --- 3rd Check: Completeness ---
    print(f"\n3. Completeness check:")
    for task in TASK_NAMES:
        task_count = file_info_df[file_info_df['task'] == task].shape[0]
        expected = len(data_dict) * N_TRIALS
        status = "✓" if task_count == expected else "✗"
        print(f"   {status} {task:15s}: {task_count} files "
              f"(expected {expected})")
    
    # --- 4th Check: Missing subjects ---
    loaded_subjects = sorted(data_dict.keys())
    print(f"\n4. Subject IDs loaded: {loaded_subjects[0]}-{loaded_subjects[-1]} "
          f"({len(loaded_subjects)} total)")
    
    if len(loaded_subjects) < N_SUBJECTS:
        all_subjects = set(range(1, N_SUBJECTS + 1))
        missing = all_subjects - set(loaded_subjects)
        if missing:
            print(f"   Missing subjects: {sorted(missing)}")
    
    # --- 5th Check: Signal amplitude range ---
    print(f"\n5. Signal amplitude statistics:")
    print(f"   Global min: {file_info_df['min'].min():.4f}")
    print(f"   Global max: {file_info_df['max'].max():.4f}")
    print(f"   Mean amplitude: {file_info_df['mean'].mean():.4f}")
    print(f"   Mean std dev:   {file_info_df['std'].mean():.4f}")
    
    # --- 6th Check: Data key used in .mat files ---
    print(f"\n6. Data key in .mat files: {file_info_df['data_key'].unique()}")
    
    # --- 7th Check: One example file in detail ---
    print(f"\n7. Example file detail:")
    example = file_info_df.iloc[0]
    print(f"   File:     {example['filename']}")
    print(f"   Subject:  {example['subject']}")
    print(f"   Task:     {example['task']}")
    print(f"   Trial:    {example['trial']}")
    print(f"   Shape:    ({example['n_channels']}, {example['n_samples']})")
    print(f"   Duration: {example['n_samples']/SAMPLING_RATE:.2f} seconds")
    
    return file_info_df

## Stress Labels Table

Creating a structured DataFrame of stress labels for every trial.

After each trial, subjects rated their stress on a 1-10 scale:
- 1 = "felt almost no stress"
- 10 = "felt extremely stressed"

Relaxation trials have no rating.

We bin the ratings into 3 categories for classification:
- Relaxed   (rating 0-3): minimal cognitive load
- Low Stress (rating 4-6): moderate cognitive effort
- High Stress (rating 7-10): significant mental strain

These thresholds are commonly used in the EEG stress literature. They roughly divide the 1-10 scale into thirds.

In [ ]:

def build_labels():

    # Returns: 
    #   labels_df with columns: subject, task, trial, rating, stress_level

    labels = []
    task_order = ['Stroop', 'Arithmetic', 'Mirror_Image']
    
    for subject_id in range(1, N_SUBJECTS + 1):
        # Relaxation trials: always labeled as 'Relaxed'
        for trial in range(1, N_TRIALS + 1):
            labels.append({
                'subject': subject_id,
                'task': 'Relaxation',
                'trial': trial,
                'rating': 0,
                'stress_level': 'Relaxed'
            })
        
        # Cognitive task trials: from Table 1 in the "Data in Brief" paper
        if subject_id in STRESS_RATINGS:
            trials_data = STRESS_RATINGS[subject_id]
            for trial_idx, trial_ratings in enumerate(trials_data):
                trial_num = trial_idx + 1
                for task_idx, rating in enumerate(trial_ratings):
                    # Bin into stress categories
                    if rating <= 3:
                        stress_level = 'Relaxed'
                    elif rating <= 6:
                        stress_level = 'Low Stress'
                    else:
                        stress_level = 'High Stress'
                    
                    labels.append({
                        'subject': subject_id,
                        'task': task_order[task_idx],
                        'trial': trial_num,
                        'rating': rating,
                        'stress_level': stress_level
                    })
    
    labels_df = pd.DataFrame(labels)
    
    print(f"\n{'='*60}")
    print(f"STRESS LABELS SUMMARY")
    print(f"{'='*60}")
    print(f"Total labeled trials: {len(labels_df)}")
    print(f"\nStress level distribution (all trials including relaxation):")
    level_counts = labels_df['stress_level'].value_counts()
    for level in ['Relaxed', 'Low Stress', 'High Stress']:
        count = level_counts.get(level, 0)
        pct = count / len(labels_df) * 100
        print(f"  {level:12s}: {count:4d} trials ({pct:.1f}%)")
    
    print(f"\nMean stress rating by task (cognitive tasks only):")
    cognitive = labels_df[labels_df['task'] != 'Relaxation']
    for task in task_order:
        task_data = cognitive[cognitive['task'] == task]
        print(f"  {task:15s}: {task_data['rating'].mean():.2f} "
              f"(std={task_data['rating'].std():.2f})")
    
    return labels_df

## Visualizations

### Plotting a raw EEG signal for a Subject

Plot raw EEG signals from all 4 tasks for the same subject.

- Relaxation: smoother, more rhythmic waves (alpha activity ~10 Hz)
- Stress tasks: more irregular, higher-frequency activity (beta ~20 Hz)
- Each channel captures a slightly different perspective of brain activity

In [9]:
def plot_1_raw_eeg_signals(data_dict, output_dir):
    
    # Find a subject with all 4 tasks
    subject_id = None
    for sid in sorted(data_dict.keys()):
        if all(t in data_dict[sid] for t in TASK_NAMES):
            subject_id = sid
            break
    if subject_id is None:
        subject_id = min(data_dict.keys())
    
    print(f"\nPlot 1: Raw EEG signals (Subject {subject_id})")
    
    # Selecting 5 key channels across different brain regions
    # Gives a good spatial overview from front to back
    key_channels = {
        'Fp1': 2,   # Frontopolar: forehead, picks up eye artifacts
        'Fz':  1,   # Frontal midline: good for stress/attention
        'Cz':  0,   # Central midline: motor and cognitive
        'Pz': 16,   # Parietal midline: spatial processing, alpha
        'Oz': 17,   # Occipital midline: visual processing
    }
    
    fig, axes = plt.subplots(len(TASK_NAMES), 1, figsize=(16, 12), sharex=True)
    fig.suptitle(f'Raw EEG Signals Across 4 Tasks — Subject {subject_id}\n'
                 f'5 key channels shown (Fp1→Fz→Cz→Pz→Oz, front to back)',
                 fontsize=14, fontweight='bold', y=0.98)
    
    for task_idx, task_name in enumerate(TASK_NAMES):
        ax = axes[task_idx]
        color = TASK_COLORS[task_name]
        
        if task_name not in data_dict[subject_id]:
            ax.text(0.5, 0.5, f'{task_name}: no data', transform=ax.transAxes,
                    ha='center', fontsize=12, color='gray')
            continue
        
        trial_num = min(data_dict[subject_id][task_name].keys())
        eeg = data_dict[subject_id][task_name][trial_num]
        n_samples = eeg.shape[1] if eeg.ndim > 1 else len(eeg)
        time = np.arange(n_samples) / SAMPLING_RATE
        
        # Plot each key channel with a vertical offset
        for i, (ch_name, ch_idx) in enumerate(key_channels.items()):
            if ch_idx < eeg.shape[0]:
                signal = eeg[ch_idx]
                # Normalize for display (z-score) and add vertical offset
                signal_norm = (signal - signal.mean()) / (signal.std() + 1e-8)
                offset = i * 5  # Space channels apart vertically
                ax.plot(time, signal_norm + offset, color=color,
                        linewidth=0.5, alpha=0.8)
        
        ax.set_ylabel(task_name.replace('_', '\n'), fontsize=11,
                      fontweight='bold', rotation=0, labelpad=65, va='center')
        ax.set_yticks([i * 5 for i in range(len(key_channels))])
        ax.set_yticklabels(list(key_channels.keys()), fontsize=9)
        ax.set_xlim(0, min(TRIAL_DURATION, time[-1]))
        ax.tick_params(axis='y', length=0)
    
    axes[-1].set_xlabel('Time (seconds)', fontsize=12)
    
    # Add annotation explaining what we see
    fig.text(0.5, 0.01,
             'Note: Each line is one EEG channel (z-score normalized). '
             'Compare the signal patterns between Relaxation (smoother) and stress tasks (more irregular).',
             ha='center', fontsize=9, color='gray', style='italic')
    
    plt.tight_layout(rect=[0.08, 0.03, 1, 0.95])
    plt.savefig(os.path.join(output_dir, '01_raw_eeg_all_tasks.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: 01_raw_eeg_all_tasks.png")

### Zoomed Comparison

Zoomed in view of 2 seconds of EEG from Relaxation vs Arithmetic on the Fz channel (frontal midline).

Why Fz channel?
* Fz sits right at the front-center of the head. Frontal regions are strongly involved in executive functions, working memory, and cognitive control which are exactly what's engaged during stress tasks. Research shows that beta power (13-30 Hz) at frontal sites increases during mental stress, while alpha power (8-13 Hz) decreases.

In [10]:

def plot_2_zoomed_comparison(data_dict, output_dir):
    subject_id = None
    for sid in sorted(data_dict.keys()):
        if 'Relaxation' in data_dict[sid] and 'Arithmetic' in data_dict[sid]:
            subject_id = sid
            break
    if subject_id is None:
        return
    
    print(f"\nPlot 2: Zoomed comparison Relaxation vs Arithmetic (Subject {subject_id})")
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True, sharey=True)
    fig.suptitle(f'Fz Channel: Relaxation vs Arithmetic (2-second window)\n'
                 f'Subject {subject_id} — Look for frequency differences',
                 fontsize=13, fontweight='bold')
    
    ch_idx = 1  # Fz
    zoom_start = 5 * SAMPLING_RATE   # Start at 5 seconds
    zoom_end = 7 * SAMPLING_RATE     # End at 7 seconds
    
    for idx, (task, color) in enumerate([('Relaxation', '#2ecc71'), 
                                          ('Arithmetic', '#3498db')]):
        ax = axes[idx]
        trial_num = min(data_dict[subject_id][task].keys())
        eeg = data_dict[subject_id][task][trial_num]
        
        signal = eeg[ch_idx, zoom_start:zoom_end]
        time = np.arange(len(signal)) / SAMPLING_RATE + 5  # Start from 5s
        
        ax.plot(time, signal, color=color, linewidth=1.2)
        ax.fill_between(time, signal, alpha=0.15, color=color)
        ax.set_ylabel(f'{task}\n(μV)', fontsize=11, fontweight='bold')
        ax.set_title(f'{"Smooth, rhythmic alpha waves (~10 Hz)" if task == "Relaxation" else "Faster, irregular beta activity (~20 Hz)"}',
                     fontsize=10, color='gray', style='italic', loc='left')
    
    axes[-1].set_xlabel('Time (seconds)', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '02_zoomed_relax_vs_arithmetic.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: 02_zoomed_relax_vs_arithmetic.png")

### Stress Labels

Visualize the distribution of self-reported stress ratings. Shows us class balance and which tasks are most stressful.

In [ ]:

def plot_3_stress_labels(labels_df, output_dir):
     
    print(f"\nPlot 3: Stress label distributions")
    
    cognitive = labels_df[labels_df['task'] != 'Relaxation'].copy()
    
    fig = plt.figure(figsize=(16, 10))
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)
    fig.suptitle('Self-Reported Stress Ratings — SAM 40 Dataset\n'
                 '40 subjects rated their stress 1-10 after each task trial',
                 fontsize=14, fontweight='bold', y=0.98)
    
    # --- Panel A: Overall rating histogram ---
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.hist(cognitive['rating'], bins=range(1, 12), color='#3498db',
             edgecolor='white', alpha=0.8, align='left', rwidth=0.85)
    ax1.set_xlabel('Stress Rating (1-10)')
    ax1.set_ylabel('Count')
    ax1.set_title('A. Overall rating distribution', fontweight='bold', fontsize=11)
    ax1.set_xticks(range(1, 11))
    
    # --- Panel B: Rating by task (overlapping histograms) ---
    ax2 = fig.add_subplot(gs[0, 1])
    for task in ['Stroop', 'Arithmetic', 'Mirror_Image']:
        task_ratings = cognitive[cognitive['task'] == task]['rating']
        ax2.hist(task_ratings, bins=range(1, 12), alpha=0.45,
                 label=task.replace('_', ' '), color=TASK_COLORS[task],
                 edgecolor='white', align='left', rwidth=0.85)
    ax2.set_xlabel('Stress Rating (1-10)')
    ax2.set_ylabel('Count')
    ax2.set_title('B. Ratings by task type', fontweight='bold', fontsize=11)
    ax2.legend(fontsize=9)
    ax2.set_xticks(range(1, 11))
    
    # --- Panel C: Box plot by task ---
    ax3 = fig.add_subplot(gs[0, 2])
    task_order_plot = ['Stroop', 'Arithmetic', 'Mirror_Image']
    bp_data = [cognitive[cognitive['task'] == t]['rating'] for t in task_order_plot]
    bp = ax3.boxplot(bp_data, labels=[t.replace('_', '\n') for t in task_order_plot],
                     patch_artist=True, widths=0.6)
    for patch, task in zip(bp['boxes'], task_order_plot):
        patch.set_facecolor(TASK_COLORS[task])
        patch.set_alpha(0.5)
    ax3.set_ylabel('Stress Rating')
    ax3.set_title('C. Rating spread by task', fontweight='bold', fontsize=11)
    
    # --- Panel D: 3 class distribution (pie chart) ---
    ax4 = fig.add_subplot(gs[1, 0])
    level_counts = labels_df['stress_level'].value_counts()
    level_order = ['Relaxed', 'Low Stress', 'High Stress']
    counts = [level_counts.get(l, 0) for l in level_order]
    colors_pie = ['#2ecc71', '#f39c12', '#e74c3c']
    wedges, texts, autotexts = ax4.pie(counts, labels=level_order, colors=colors_pie,
                                        autopct='%1.1f%%', startangle=90,
                                        textprops={'fontsize': 10})
    ax4.set_title('D. Stress level categories\n(all trials incl. relaxation)',
                  fontweight='bold', fontsize=11)
    
    # --- Panel E: Mean rating per subject ---
    ax5 = fig.add_subplot(gs[1, 1:])
    subj_means = cognitive.groupby('subject')['rating'].mean().sort_values()
    bars = ax5.bar(range(len(subj_means)), subj_means.values,
                   color='#9b59b6', alpha=0.7, edgecolor='white')
    ax5.set_xlabel('Subject (sorted by mean stress rating)')
    ax5.set_ylabel('Mean Stress Rating')
    ax5.set_title('E. Average stress rating per subject — shows individual variability',
                  fontweight='bold', fontsize=11)
    ax5.axhline(y=cognitive['rating'].mean(), color='red', linestyle='--',
                alpha=0.7, label=f'Grand mean: {cognitive["rating"].mean():.1f}')
    ax5.legend(fontsize=9)
    
    # Remove x-ticks for cleaner look (40 subject IDs would be cluttered)
    ax5.set_xticks([])
    
    plt.savefig(os.path.join(output_dir, '03_stress_label_distribution.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: 03_stress_label_distribution.png")

### Channel Layout

Shows where the 32 electrodes sit on the scalp.

The 10-20 system:
* This is the international standard for placing EEG electrodes. The name "10-20" means electrodes are spaced at 10% or 20% of the total distance between skull landmarks.

Different brain regions process different things. When we later look at which channels are most useful for classifying stress, we can map that back to brain regions. For example, if frontal channels (F3, F4, Fz) are most important, that tells us stress detection relies on executive function areas, which makes neuroscientific sense.

In [14]:

def plot_4_channel_layout(output_dir):
    
    print(f"\nPlot 4: Channel layout (10-20 system)")
    
    # 2D approximate positions (top-down view of the head)
    channel_pos = {
        'Fp1':(-0.3,0.9),'Fp2':(0.3,0.9),
        'F7':(-0.7,0.6),'F3':(-0.35,0.6),'Fz':(0,0.6),'F4':(0.35,0.6),'F8':(0.7,0.6),
        'FT9':(-0.9,0.4),'FC5':(-0.55,0.4),'FC1':(-0.2,0.4),
        'FC2':(0.2,0.4),'FC6':(0.55,0.4),'FT10':(0.9,0.4),
        'T7':(-0.9,0.0),'C3':(-0.45,0.0),'Cz':(0,0.0),'C4':(0.45,0.0),'T8':(0.9,0.0),
        'CP5':(-0.55,-0.3),'CP1':(-0.2,-0.3),'CP2':(0.2,-0.3),'CP6':(0.55,-0.3),
        'P7':(-0.7,-0.5),'P3':(-0.35,-0.5),'Pz':(0,-0.5),'P4':(0.35,-0.5),'P8':(0.7,-0.5),
        'PO9':(-0.6,-0.7),'PO10':(0.6,-0.7),
        'O1':(-0.25,-0.85),'Oz':(0,-0.85),'O2':(0.25,-0.85),
    }
    
    # Color by brain region
    region_colors = {
        'Fp':'#ff6b6b','F':'#ffa07a','FC':'#ffcc5c','FT':'#ffcc5c',
        'C':'#88d8b0','T':'#77b5fe','CP':'#b2d8b2',
        'P':'#dda0dd','PO':'#d8bfd8','O':'#cd853f',
    }
    
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_title('32-Channel EEG Electrode Placement\n'
                 'SAM 40 Dataset — Emotiv EPOC Flex (10-20 System)',
                 fontsize=14, fontweight='bold')
    
    # Draw head outline
    theta = np.linspace(0, 2*np.pi, 100)
    ax.plot(np.cos(theta), np.sin(theta), 'k-', linewidth=2)
    ax.plot([-0.1,0,0.1], [1.0,1.12,1.0], 'k-', linewidth=2)  # Nose
    ax.plot([-1.05,-1.1,-1.1,-1.05], [-0.1,-0.05,0.05,0.1], 'k-', linewidth=2)  # Left ear
    ax.plot([1.05,1.1,1.1,1.05], [-0.1,-0.05,0.05,0.1], 'k-', linewidth=2)  # Right ear
    
    for ch_name, (x, y) in channel_pos.items():
        color = '#cccccc'
        for prefix, c in sorted(region_colors.items(), key=lambda x: -len(x[0])):
            if ch_name.startswith(prefix):
                color = c
                break
        ax.scatter(x, y, s=500, c=color, edgecolors='black', linewidth=1.5, zorder=5)
        ax.annotate(ch_name, (x, y), ha='center', va='center', fontsize=7,
                    fontweight='bold', zorder=6)
    
    # Legend
    legend_items = [
        ('Frontopolar (Fp)','#ff6b6b'), ('Frontal (F)','#ffa07a'),
        ('Fronto-Central (FC/FT)','#ffcc5c'), ('Central (C)','#88d8b0'),
        ('Temporal (T)','#77b5fe'), ('Centro-Parietal (CP)','#b2d8b2'),
        ('Parietal (P)','#dda0dd'), ('Parieto-Occipital (PO)','#d8bfd8'),
        ('Occipital (O)','#cd853f'),
    ]
    for label, color in legend_items:
        ax.scatter([], [], c=color, edgecolors='black', s=100, label=label)
    ax.legend(loc='lower left', fontsize=8, framealpha=0.9)
    
    for text, pos in [('FRONT (Nose)',(0,1.2)), ('BACK',(0,-1.1)),
                       ('LEFT',(-1.25,0)), ('RIGHT',(1.25,0))]:
        rot = 90 if 'LEFT' in text else (-90 if 'RIGHT' in text else 0)
        ax.annotate(text, pos, ha='center', va='center', fontsize=11,
                    fontweight='bold', color='gray', rotation=rot)
    
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.3, 1.4)
    ax.set_aspect('equal')
    ax.axis('off')
    
    plt.savefig(os.path.join(output_dir, '04_channel_layout.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: 04_channel_layout.png")

### Data Quality

Data quality dashboard: shapes, amplitudes, completeness.
    
This plot surfaces any anomalies in the dataset before spending hours on feature extraction.

In [15]:
def plot_5_data_quality(file_info_df, output_dir):
    print(f"\nPlot 5: Data quality dashboard")
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Data Quality Dashboard — SAM 40 Dataset',
                 fontsize=14, fontweight='bold')
    
    # --- Panel A: Sample count distribution ---
    ax = axes[0, 0]
    ax.hist(file_info_df['n_samples'], bins=30, color='#3498db',
            edgecolor='white', alpha=0.8)
    ax.axvline(EXPECTED_SAMPLES, color='red', linestyle='--', linewidth=2,
               label=f'Expected ({EXPECTED_SAMPLES})')
    ax.set_xlabel('Number of Samples per Trial')
    ax.set_ylabel('Count')
    ax.set_title('A. Sample count distribution', fontweight='bold')
    ax.legend()
    
    # --- Panel B: Signal amplitude by task ---
    ax = axes[0, 1]
    for task in TASK_NAMES:
        task_data = file_info_df[file_info_df['task'] == task]
        ax.scatter(task_data['mean'], task_data['std'],
                   c=TASK_COLORS[task], alpha=0.5, s=30,
                   label=task.replace('_', ' '))
    ax.set_xlabel('Mean Amplitude')
    ax.set_ylabel('Standard Deviation')
    ax.set_title('B. Signal statistics by task', fontweight='bold')
    ax.legend(fontsize=8)
    
    # --- Panel C: Completeness heatmap ---
    ax = axes[1, 0]
    n_subjects = file_info_df['subject'].nunique()
    completeness = np.zeros((n_subjects, len(TASK_NAMES)))
    subject_ids = sorted(file_info_df['subject'].unique())
    
    for _, row in file_info_df.iterrows():
        subj_idx = subject_ids.index(row['subject'])
        task_idx = TASK_NAMES.index(row['task']) if row['task'] in TASK_NAMES else -1
        if task_idx >= 0:
            completeness[subj_idx, task_idx] += 1
    
    im = ax.imshow(completeness.T, aspect='auto', cmap='YlOrRd', vmin=0, vmax=3)
    ax.set_xlabel('Subject Index')
    ax.set_ylabel('Task')
    ax.set_yticks(range(len(TASK_NAMES)))
    ax.set_yticklabels([t.replace('_', ' ') for t in TASK_NAMES], fontsize=9)
    ax.set_title('C. Trials per subject per task (expect 3)',
                 fontweight='bold')
    plt.colorbar(im, ax=ax, label='Trial Count', shrink=0.8)
    
    # --- Panel D: Signal power per subject ---
    ax = axes[1, 1]
    subj_power = file_info_df.groupby('subject')['std'].mean()
    colors = ['#e74c3c' if v > subj_power.mean() + 2*subj_power.std() or 
              v < subj_power.mean() - 2*subj_power.std() else '#3498db' 
              for v in subj_power.values]
    ax.bar(subj_power.index, subj_power.values, color=colors, alpha=0.7)
    ax.axhline(subj_power.mean(), color='black', linestyle='--', alpha=0.5,
               label=f'Mean: {subj_power.mean():.2f}')
    ax.set_xlabel('Subject ID')
    ax.set_ylabel('Mean Signal Std Dev')
    ax.set_title('D. Per-subject signal power\n(red = outlier >2σ from mean)',
                 fontweight='bold')
    ax.legend(fontsize=9)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '05_data_quality_dashboard.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: 05_data_quality_dashboard.png")

## MAIN

In [18]:
if __name__ == "__main__":
    print("=" * 60)
    print("  EEG COGNITIVE STRESS CLASSIFICATION")
    print("  Phase 1: Data Loading & Exploration")
    print("=" * 60)

  EEG COGNITIVE STRESS CLASSIFICATION
  Phase 1: Data Loading & Exploration


In [21]:
# Load all Data

print(f"\n▶ Loading data from: {DATA_DIR}")
data_dict, file_info_df = load_all_data(DATA_DIR)
    
if data_dict is None:
    print("\n⚠ Could not load data. Please check DATA_DIR path.")
    print(f"  Current path: {os.path.abspath(DATA_DIR)}")
    print(f"  Expected structure: {DATA_DIR}/filtered_data/*.mat")
    exit(1)


▶ Loading data from: ..\Data\SAM40
Found 480 .mat files in filtered_data/
  Loading file 1/480: Arithmetic_sub_10_trial1.mat
  Loading file 50/480: Arithmetic_sub_25_trial2.mat
  Loading file 100/480: Arithmetic_sub_40_trial1.mat
  Loading file 150/480: Mirror_image_sub_19_trial3.mat
  Loading file 200/480: Mirror_image_sub_34_trial2.mat
  Loading file 250/480: Relax_sub_13_trial1.mat
  Loading file 300/480: Relax_sub_28_trial3.mat
  Loading file 350/480: Relax_sub_6_trial2.mat
  Loading file 400/480: Stroop_sub_22_trial1.mat
  Loading file 450/480: Stroop_sub_37_trial3.mat

LOADING COMPLETE
Subjects loaded:  40
Files loaded:     480
Files expected:   480


In [22]:
# Exploring Structure

print(f"\n▶ Exploring data structure...")
explore_structure(data_dict, file_info_df)


▶ Exploring data structure...

DATA STRUCTURE EXPLORATION

1. Channel counts: [32]
   ✓ All files have exactly 32 channels

2. Sample counts:
   Min:  3200
   Max:  3200
   Mean: 3200.0
   Expected: 3200
   Files with exactly 3200 samples: 480/480

3. Completeness check:
   ✓ Relaxation     : 120 files (expected 120)
   ✓ Stroop         : 120 files (expected 120)
   ✓ Mirror_Image   : 120 files (expected 120)
   ✓ Arithmetic     : 120 files (expected 120)

4. Subject IDs loaded: 1-40 (40 total)

5. Signal amplitude statistics:
   Global min: -66.4404
   Global max: 80.2733
   Mean amplitude: -0.0143
   Mean std dev:   7.1877

6. Data key in .mat files: ['Clean_data']

7. Example file detail:
   File:     Arithmetic_sub_10_trial1.mat
   Subject:  10
   Task:     Arithmetic
   Trial:    1
   Shape:    (32, 3200)
   Duration: 25.00 seconds


,filename,subject,task,trial,n_channels,n_samples,data_key,min,max,mean,std
0,Arithmetic_sub_10_trial1.mat,10,Arithmetic,1,32,3200,Clean_data,-29.956907,41.114404,-0.038035,5.611022
1,Arithmetic_sub_10_trial2.mat,10,Arithmetic,2,32,3200,Clean_data,-27.515612,37.137554,0.065793,6.053796
2,Arithmetic_sub_10_trial3.mat,10,Arithmetic,3,32,3200,Clean_data,-31.269535,39.028708,0.018959,5.954736
3,Arithmetic_sub_11_trial1.mat,11,Arithmetic,1,32,3200,Clean_data,-37.310002,47.735630,0.092546,7.977746
4,Arithmetic_sub_11_trial2.mat,11,Arithmetic,2,32,3200,Clean_data,-37.626253,44.575027,0.006973,8.241062
...,...,...,...,...,...,...,...,...,...,...,...
475,Stroop_sub_8_trial2.mat,8,Stroop,2,32,3200,Clean_data,-25.641090,26.862398,0.008636,6.374609
476,Stroop_sub_8_trial3.mat,8,Stroop,3,32,3200,Clean_data,-29.056022,32.303924,-0.001687,7.081458
477,Stroop_sub_9_trial1.mat,9,Stroop,1,32,3200,Clean_data,-26.479828,35.399520,0.008644,4.720792
478,Stroop_sub_9_trial2.mat,9,Stroop,2,32,3200,Clean_data,-28.262881,40.301421,-0.044275,5.208957


In [23]:
# Building stress labels

print(f"\n▶ Building stress labels...")
labels_df = build_labels()
    
# Save labels for later phases
labels_df.to_csv(os.path.join(OUTPUT_DIR, 'stress_labels.csv'), index=False)
print(f"\n  Labels saved to: {OUTPUT_DIR}/stress_labels.csv")


▶ Building stress labels...

STRESS LABELS SUMMARY
Total labeled trials: 480

Stress level distribution (all trials including relaxation):
  Relaxed     :  222 trials (46.2%)
  Low Stress  :  188 trials (39.2%)
  High Stress :   70 trials (14.6%)

Mean stress rating by task (cognitive tasks only):
  Stroop         : 4.05 (std=1.90)
  Arithmetic     : 5.44 (std=2.02)
  Mirror_Image   : 4.94 (std=1.90)

  Labels saved to: ..\Results\phase1/stress_labels.csv


In [24]:
# Generating Visualisations

print(f"\n▶ Generating visualizations...")
    
plot_1_raw_eeg_signals(data_dict, OUTPUT_DIR)
plot_2_zoomed_comparison(data_dict, OUTPUT_DIR)
plot_3_stress_labels(labels_df, OUTPUT_DIR)
plot_4_channel_layout(OUTPUT_DIR)
plot_5_data_quality(file_info_df, OUTPUT_DIR)


▶ Generating visualizations...

Plot 1: Raw EEG signals (Subject 1)
  ✓ Saved: 01_raw_eeg_all_tasks.png

Plot 2: Zoomed comparison Relaxation vs Arithmetic (Subject 1)
  ✓ Saved: 02_zoomed_relax_vs_arithmetic.png

Plot 3: Stress label distributions
  ✓ Saved: 03_stress_label_distribution.png

Plot 4: Channel layout (10-20 system)
  ✓ Saved: 04_channel_layout.png

Plot 5: Data quality dashboard
  ✓ Saved: 05_data_quality_dashboard.png


## Summary

In [25]:

print(f"\n{'='*60}")
print(f"  PHASE 1 COMPLETE!")
print(f"{'='*60}")
print(f"  Subjects:    {len(data_dict)}")
print(f"  Total files: {len(file_info_df)}")
print(f"  Channels:    {file_info_df['n_channels'].iloc[0]}")
print(f"  Samples:     ~{int(file_info_df['n_samples'].mean())}")
print(f"  Duration:    {file_info_df['n_samples'].mean()/SAMPLING_RATE:.1f}s per trial")
print(f"\n  Outputs saved to: {os.path.abspath(OUTPUT_DIR)}/")
print(f"    01_raw_eeg_all_tasks.png")
print(f"    02_zoomed_relax_vs_arithmetic.png")
print(f"    03_stress_label_distribution.png")
print(f"    04_channel_layout.png")
print(f"    05_data_quality_dashboard.png")
print(f"    stress_labels.csv")
print(f"\n  Next step: Phase 2 (Preprocessing)")
print(f"{'='*60}")


  PHASE 1 COMPLETE!
  Subjects:    40
  Total files: 480
  Channels:    32
  Samples:     ~3200
  Duration:    25.0s per trial

  Outputs saved to: c:\Users\hibro\OneDrive\Desktop\Desktop_Files\Projects\Python\ML_Models\Cognitive_Stress_Classification\EEG-Stress-Classification\Results\phase1/
    01_raw_eeg_all_tasks.png
    02_zoomed_relax_vs_arithmetic.png
    03_stress_label_distribution.png
    04_channel_layout.png
    05_data_quality_dashboard.png
    stress_labels.csv

  Next step: Phase 2 (Preprocessing)
